# 02 — Training (EfficientNet-B0 and ResNet-18)

**Kaggle:** Add Input → *140k Real and Fake Faces*; Accelerator = GPU; Internet = On; then *Save Version → Save & Run All*.

**Colab:** Runtime → Change runtime type → GPU.

Theory: [`docs/03_model_architecture.md`](../docs/03_model_architecture.md), [`docs/04_training.md`](../docs/04_training.md)

In [ ]:
# --- environment setup: works on Kaggle, Colab and locally ---
import sys, os
REPO = "https://github.com/Mohd2Shaan/Deepfake-Detection-Project.git"
ON_KAGGLE = os.path.exists("/kaggle/input")
ON_COLAB = "google.colab" in sys.modules
if ON_KAGGLE or ON_COLAB:
    PROJECT = "/kaggle/working/project" if ON_KAGGLE else "/content/project"
    if os.path.exists(PROJECT):
        !git -C {PROJECT} pull -q
    else:
        !git clone -q {REPO} {PROJECT}
    %cd {PROJECT}
    EXTRA = "" if ON_KAGGLE else "kagglehub"
    !pip install -q grad-cam {EXTRA}
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("working dir:", os.getcwd())

In [ ]:
# --- dataset: Kaggle -> attached input is found automatically; Colab/local -> downloaded once (~4 GB) ---
from src import config
if ON_KAGGLE:
    for ckpt in config.import_kaggle_checkpoints():      # notebooks 03/04: models from notebook 02's output
        print("trained model:", ckpt)
if not (config.DATA_DIR / "train").exists():
    if ON_KAGGLE:
        raise RuntimeError("Dataset not found - attach '140k Real and Fake Faces' with Add Input")
    !python scripts/download_dataset.py --move
print("data dir:", config.DATA_DIR)

In [ ]:
# --- checkpoints: Colab -> Google Drive (survives disconnects); Kaggle/local -> checkpoints/ ---
import sys
CKPT_DIR = "checkpoints"
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    CKPT_DIR = "/content/drive/MyDrive/deepfake-detection/checkpoints"
    !mkdir -p {CKPT_DIR}
print("checkpoints ->", CKPT_DIR)

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## Model A — EfficientNet-B0
3 frozen epochs (head only, lr 1e-4) → 7 fine-tune epochs (last 2 blocks, lr 1e-5).

If Colab disconnects, re-run the setup cells and add `--resume`.

In [ ]:
!python -m src.train --model efficientnet_b0 --checkpoint-dir {CKPT_DIR} --num-workers 4

## Model B — ResNet-18

In [ ]:
!python -m src.train --model resnet18 --checkpoint-dir {CKPT_DIR} --num-workers 4

## Training curves

In [ ]:
from IPython.display import Image as Show, display
for name in ['efficientnet', 'resnet']:
    display(Show(f'outputs/plots/training_curves_{name}.png'))

In [ ]:
# copy best checkpoints back into the project folder for evaluation / the app
!mkdir -p checkpoints && cp {CKPT_DIR}/*_best.pth checkpoints/ 2>/dev/null; ls -lh checkpoints

## Package trained models for download
Download `results.zip` from the Output tab — needed for the Streamlit app on your laptop.

In [ ]:
!zip -qr results.zip outputs checkpoints/efficientnet_best.pth checkpoints/resnet_best.pth
!ls -lh results.zip

## Observations

_Note convergence speed, gap between train/valid, effect of unfreezing at epoch 4._